# Rotations And Orientation Representations

A crystal orientation is one geometric object. The numbers that denote it are many, and
which set of numbers you meet depends entirely on who handed them to you:

- an EBSD system gives you **Bunge Euler angles**, three numbers in degrees;
- a misorientation is quoted as an **axis-angle pair**, "60 degrees about $\langle 111 \rangle$";
- a rolling-texture component is *named* by two index triples, $(hkl)[uvw]$;
- a pattern-matching dictionary is indexed on **cubochoric** coordinates, which most
  people have never heard of;
- and underneath all of them PyTex stores a **unit quaternion**, because that is the one
  representation with no singularities and no convention branches.

None of these is more correct than the others. They are charts on the same manifold, and
each exists because it makes one particular job easy and the others hard. This tutorial
is about knowing which is which.

By the end you will be able to answer, for any rotation:

1. What *is* a rotation, and what does the active convention commit you to?
2. What does each representation look like, numerically and geometrically?
3. Why does uniform sampling of Euler angles produce a badly non-uniform set of
   orientations, and what fixes it?
4. Where does each representation break, and what does PyTex do about it?
5. How do you get *all* of them from one call, in one line?

Everything below is computed live. Where a number has an analytic value, the notebook
computes it from the code and compares.

## 0. Setup

Two frames and one phase. Nickel, from the pinned CIF fixture, is used in the sections
that need a lattice; the rotation sections need no crystal at all, which is itself a
distinction worth keeping straight.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from pytex import (
    CUBOCHORIC_CUBE_HALF_EDGE,
    HOMOCHORIC_BALL_RADIUS,
    Orientation,
    OrientationRepresentationSet,
    RepresentationKind,
    Rotation,
    convert_orientations,
    crystal_frame,
    cubochoric_from_quaternions,
    get_phase_fixture,
    homochoric_from_quaternions,
    ideal_orientation_indices,
    orientation_representations,
    quaternions_from_cubochoric,
    quaternions_from_euler_angles,
    quaternions_to_euler_angles,
    rotation_representations,
    specimen_frame,
)

CRYSTAL = crystal_frame()
SPECIMEN = specimen_frame()

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz.*")
    warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF.*")
    NICKEL = get_phase_fixture("ni_fcc").load_phase(crystal_frame=CRYSTAL)

print(f"{NICKEL.name}: a = {NICKEL.lattice.a:.4f} A, point group {NICKEL.symmetry.point_group}")
print(f"homochoric ball radius  R1 = {HOMOCHORIC_BALL_RADIUS:.6f}")
print(f"cubochoric cube half-edge  = {CUBOCHORIC_CUBE_HALF_EDGE:.6f}")

## 1. What a rotation is

A rotation is a linear map of three-dimensional space that preserves lengths, angles, and
handedness. As a matrix that means

$$R^{\mathsf{T}}R = I, \qquad \det R = +1,$$

the group $SO(3)$. The second condition is the one that does work: dropping it admits
reflections, which preserve lengths and angles but turn a right-handed crystal into a
left-handed one. PyTex rejects an improper matrix at construction rather than letting a
reflection propagate.

PyTex uses the **active** convention throughout: $R$ maps a vector to its rotated image,
$v' = R v$, both expressed in one fixed frame. The passive reading — same vector, new
frame — is the transpose, and mixing the two is the single most common source of
orientations that are wrong by exactly an inverse.

In [ ]:
rotation = Rotation.from_axis_angle((0.0, 0.0, 1.0), np.radians(30.0))
matrix = rotation.as_matrix()

print("R^T R - I max deviation:", float(np.abs(matrix.T @ matrix - np.eye(3)).max()))
print("det R:", float(np.linalg.det(matrix)))

v = np.array([1.0, 0.0, 0.0])
print("R v =", np.round(matrix @ v, 6), " (x rotated 30 deg towards y)")

try:
    Rotation.from_matrix(np.diag([1.0, 1.0, -1.0]))
except ValueError as error:
    print("\nreflection rejected:", error)

### The one picture worth having

Every rotation, no matter how it is written down, does the same thing: it turns the frame
about some axis by some angle. The figure below draws that for the rotation above.

In [ ]:
def draw_frame(ax, matrix, *, alpha=1.0, labels=None, style="-"):
    colours = ("#d62728", "#2ca02c", "#1f77b4")
    for column, colour in enumerate(colours):
        axis = matrix[:, column]
        ax.plot(*np.column_stack([np.zeros(3), axis]), style, color=colour, alpha=alpha, lw=2)
        if labels is not None:
            ax.text(*(axis * 1.12), labels[column], color=colour, alpha=alpha)


fig = plt.figure(figsize=(5.4, 5.0))
ax = fig.add_subplot(projection="3d")
draw_frame(ax, np.eye(3), alpha=0.32, labels=("x", "y", "z"), style="--")
draw_frame(ax, matrix, labels=("x'", "y'", "z'"))

axis, angle_rad = rotation.axis, np.radians(rotation.angle_deg)
ax.plot(*np.column_stack([-axis * 1.3, axis * 1.3]), color="black", lw=1.2, ls=":")
ax.set_title(f"a rotation of {rotation.angle_deg:.1f} deg about [{axis[0]:.0f} {axis[1]:.0f} {axis[2]:.0f}]"
             "\ndashed = original frame, solid = rotated, dotted = rotation axis")
ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
ax.set_box_aspect((1, 1, 1))
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
plt.show()

## 2. Axis and angle: Euler's rotation theorem

**Euler's rotation theorem.** Every rotation of three-dimensional space fixes a line. That
line is the rotation axis $\hat{n}$, and the rotation is a turn by some angle $\omega$
about it. So *two* numbers' worth of axis plus one of angle — three parameters — suffice,
which is why nine matrix entries with six orthonormality constraints was always overkill.

The axis is the eigenvector of $R$ with eigenvalue $+1$, and the angle follows from the
trace, since $\operatorname{tr} R = 1 + 2\cos\omega$:

$$\omega = \arccos\!\left(\frac{\operatorname{tr}R - 1}{2}\right).$$

PyTex reports $\omega \in [0, \pi]$ and flips the axis rather than allowing negative
angles, so that $(\hat{n}, \omega)$ and $(-\hat{n}, -\omega)$ cannot both appear for the
same rotation.

In [ ]:
test = Rotation.from_bunge_euler(35.0, 45.0, 60.0)
matrix = test.as_matrix()

trace_angle = np.degrees(np.arccos(np.clip((np.trace(matrix) - 1.0) / 2.0, -1.0, 1.0)))
eigenvalues, eigenvectors = np.linalg.eig(matrix)
fixed = np.real(eigenvectors[:, np.argmin(np.abs(eigenvalues - 1.0))])
fixed = fixed / np.linalg.norm(fixed) * np.sign(np.dot(np.real(fixed), test.axis))

print(f"angle from the trace : {trace_angle:.9f} deg")
print(f"angle from PyTex     : {test.angle_deg:.9f} deg")
print(f"axis  from eigenvector: {np.round(fixed, 9)}")
print(f"axis  from PyTex      : {np.round(test.axis, 9)}")
print(f"the axis is fixed: |R n - n| = {float(np.abs(matrix @ test.axis - test.axis).max()):.2e}")

## 3. Quaternions: the storage form

Axis-angle is geometric but awkward to compose: the axis of a product is not any simple
function of the two axes. Quaternions fix that. Define

$$q = \left(\cos\tfrac{\omega}{2},\; \hat{n}\sin\tfrac{\omega}{2}\right) = (w, x, y, z),$$

a unit vector in four dimensions. Composition of rotations becomes quaternion
multiplication, which is bilinear — 16 multiplications rather than a 27-multiplication
matrix product, and with no drift away from orthonormality to renormalise.

**The double cover.** $q$ and $-q$ denote the *same* rotation, because turning the half
angle by $\pi$ negates all four components while the rotation is unchanged. The unit
quaternions form a sphere $S^3$ that covers $SO(3)$ twice. This is not a defect to be
suppressed: it is why $SO(3)$ is not simply connected, and it is why a naive average of
quaternions can cancel to nothing.

A `Rotation` keeps whichever sign the arithmetic produced, since nothing in the algebra
cares. The *reporting* surfaces — `rotation_representations` and
`OrientationRepresentationSet` — canonicalize to $w \ge 0$, so that two reports of one
rotation can be compared component by component rather than only up to a sign.

In [ ]:
first = Rotation.from_axis_angle((0.0, 0.0, 1.0), np.radians(90.0))
second = Rotation.from_axis_angle((1.0, 0.0, 0.0), np.radians(90.0))

composed = first.compose(second)
print("quaternion product  :", np.round(composed.quaternion, 9))
print("matrix product check:", float(np.abs(
    composed.as_matrix() - first.as_matrix() @ second.as_matrix()).max()))

negated = Rotation(quaternion=-composed.quaternion)
print("\nq and -q build the same matrix:", float(np.abs(
    negated.as_matrix() - composed.as_matrix()).max()))
print("but componentwise they look 2 apart:", float(np.abs(
    negated.quaternion - composed.quaternion).max()))
print("reports therefore canonicalize to w >= 0:",
      rotation_representations(negated).quaternion)

reversed_order = second.compose(first)
print("\nrotations do not commute -- and the rotation angle alone does not reveal it:")
print(f"  first-then-second: {composed.angle_deg:.3f} deg about {np.round(composed.axis, 4)}")
print(f"  reversed order   : {reversed_order.angle_deg:.3f} deg about {np.round(reversed_order.axis, 4)}")
print(f"  the two rotations differ by {np.degrees(composed.distance_to(reversed_order)):.3f} deg")
print("\n(Rotation.distance_to returns RADIANS: it is the geodesic angle on SO(3).)")

## 4. Rodrigues and Rodrigues-Frank: where fundamental zones are polyhedra

The Rodrigues vector packs axis and angle into three numbers:

$$\boldsymbol{\rho} = \hat{n}\tan\tfrac{\omega}{2}.$$

Its virtue is structural rather than numerical: **in Rodrigues space the fundamental zone
of a crystal symmetry is a convex polyhedron**, bounded by flat planes. Testing whether an
orientation is inside the zone becomes a handful of linear inequalities, which is why
Rodrigues space is the natural home of symmetry reduction.

Its vice is the tangent. As $\omega \to \pi$, $\tan(\omega/2) \to \infty$, so a
180-degree rotation — an ordinary, physically unremarkable rotation, and a very common one
in twinning — sits at infinity, and the 3-vector loses the *axis* along with the magnitude
in the overflowing product. The homogeneous **Rodrigues-Frank** form
$(\hat{n}, \tan(\omega/2))$ keeps the two apart. Its magnitude is a projective coordinate,
so $\omega = \pi$ is the point at infinity — still a well-defined representation, and one
PyTex inverts exactly, as the cell below shows.

In [ ]:
angles_deg = np.linspace(0.0, 179.5, 400)
axis = np.array([0.0, 0.0, 1.0])
quaternions = np.column_stack([
    np.cos(np.radians(angles_deg) / 2.0),
    np.outer(np.sin(np.radians(angles_deg) / 2.0), axis),
])

rodrigues = convert_orientations(quaternions, source="quaternion", target="rodrigues")
homochoric = convert_orientations(quaternions, source="quaternion", target="homochoric")

fig, ax = plt.subplots(figsize=(6.8, 4.0))
ax.plot(angles_deg, np.linalg.norm(rodrigues, axis=1), label=r"Rodrigues  $\tan(\omega/2)$")
ax.plot(angles_deg, np.linalg.norm(homochoric, axis=1),
        label=r"homochoric  $[\frac{3}{4}(\omega-\sin\omega)]^{1/3}$")
ax.axhline(HOMOCHORIC_BALL_RADIUS, color="grey", ls="--", lw=0.9)
ax.text(4.0, HOMOCHORIC_BALL_RADIUS + 0.12, r"$R_1 = (3\pi/4)^{1/3}$", color="grey")
ax.set_xlabel("rotation angle (deg)")
ax.set_ylabel("radial coordinate")
ax.set_ylim(0.0, 6.0)
ax.set_title("Rodrigues diverges at 180 deg; the homochoric radius does not")
ax.legend()
plt.show()

twin_axis = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0)
for angle_deg in (179.999, 180.0):
    sharp = Rotation.from_axis_angle(twin_axis, np.radians(angle_deg))
    frank = sharp.to_rodrigues(frank=True)
    print(f"at {angle_deg} deg about [111]:")
    print("  Rodrigues      :", np.array2string(sharp.to_rodrigues(), precision=3))
    print("  Rodrigues-Frank:", np.array2string(frank, precision=6))
    rebuilt = convert_orientations(frank[None, :], source="rodrigues_frank", target="quaternion")
    recovered = Rotation(quaternion=rebuilt[0])
    print(f"  and Frank inverts to {recovered.angle_deg:.6f} deg about "
          f"{np.round(recovered.axis, 6)}")

print("\nAt exactly 180 deg the 3-vector has overflowed to ~1e16 and its axis is lost in")
print("the product; the Frank form still names the axis exactly, magnitude at infinity.")

## 5. Euler angles: three turns, and the convention that decides what they mean

An Euler triple prescribes three successive rotations about named axes. The two
conventions PyTex supports differ in the second axis:

| convention | sequence | angles | community |
| --- | --- | --- | --- |
| Bunge | $Z X Z$ | $(\varphi_1, \Phi, \varphi_2)$ | texture, EBSD, essentially all software |
| Matthies / ABG | $Z Y Z$ | $(\alpha, \beta, \gamma)$ | Matthies/Roe school |

The same three numbers denote **different rotations** under the two conventions. This is
why `EulerSet` carries its convention with the numbers rather than leaving it to a
docstring: a triple without a convention is not data.

In [ ]:
triple = (35.0, 45.0, 60.0)
bunge = Rotation.from_euler(*triple, convention="bunge")
zyz = Rotation.from_euler(*triple, convention="matthies")

print(f"the same triple {triple} read two ways gives two rotations "
      f"{np.degrees(bunge.distance_to(zyz)):.3f} deg apart\n")

report = rotation_representations(bunge)
print("one rotation, two triples:")
print(f"  Bunge ZXZ (phi1, Phi, phi2) = {np.round(report.euler_bunge_deg, 4)}")
print(f"  ZYZ    (alpha, beta, gamma) = {np.round(report.euler_matthies_deg, 4)}")
print(f"  and both rebuild the same matrix: "
      f"{float(np.abs(Rotation.from_euler(*report.euler_matthies_deg, convention='matthies').as_matrix() - bunge.as_matrix()).max()):.2e}")

### Gimbal lock is a property of the chart, not of the rotation

When $\Phi = 0$ the first and third Bunge rotations are about the *same* axis, so only
$\varphi_1 + \varphi_2$ is determined: a whole one-parameter family of triples denotes
one rotation. The rotation is entirely unremarkable — it is the chart that has collapsed.

PyTex resolves the ambiguity by reporting the third angle as zero. That is a choice, and
it is stated rather than hidden: a round trip through Euler angles returns the same
rotation but not necessarily the same numbers.

In [ ]:
degenerate = np.array([[30.0, 0.0, 20.0], [70.0, 0.0, -20.0], [12.5, 0.0, 37.5]])
quaternions = quaternions_from_euler_angles(degenerate, convention="bunge")
recovered = quaternions_to_euler_angles(quaternions, convention="bunge")

print("three different triples with Phi = 0:")
for original, back in zip(degenerate, recovered, strict=True):
    print(f"  {original}  ->  {np.round(back, 6)}")

pairwise = np.degrees(2.0 * np.arccos(np.clip(np.abs(quaternions @ quaternions.T), -1.0, 1.0)))
print("\npairwise angles between them (deg):\n", np.round(pairwise, 6))
print("\nAll three have phi1 + phi2 = 50 deg, and all three are one rotation.")

## 6. The measure problem, and why it matters

Here is the section that changes how people sample orientations.

The invariant (Haar) measure on $SO(3)$, written in axis-angle coordinates, is

$$\mathrm{d}\mu = \frac{1}{\pi^{2}}\,(1 - \cos\omega)\,\mathrm{d}\omega\,\mathrm{d}\Omega_{\hat{n}}.$$

The factor $(1 - \cos\omega)$ says that rotations by large angles are *far* more numerous
than rotations by small ones — there is simply more room out there. A consequence with an
exact value: the mean rotation angle of a uniform random orientation is

$$\langle \omega \rangle = \int_{0}^{\pi}\omega\,\frac{1-\cos\omega}{\pi}\,\mathrm{d}\omega
= \frac{\pi}{2} + \frac{2}{\pi} \approx 126.4756^{\circ}.$$

Now the trap. Drawing $\varphi_1, \Phi, \varphi_2$ each uniformly is the obvious thing to
do and it is **wrong**: the Bunge volume element carries a $\sin\Phi$, so uniform angles
over-weight the poles of the chart. The figure below compares three samplers against the
analytic law.

In [ ]:
rng = np.random.default_rng(20260809)
count = 200_000

naive = np.column_stack([
    rng.uniform(0.0, 360.0, count),
    rng.uniform(0.0, 180.0, count),
    rng.uniform(0.0, 360.0, count),
])
naive_q = quaternions_from_euler_angles(naive, convention="bunge")

corrected = naive.copy()
corrected[:, 1] = np.degrees(np.arccos(rng.uniform(-1.0, 1.0, count)))
corrected_q = quaternions_from_euler_angles(corrected, convention="bunge")

cube = rng.uniform(-CUBOCHORIC_CUBE_HALF_EDGE, CUBOCHORIC_CUBE_HALF_EDGE, (count, 3))
cubochoric_q = quaternions_from_cubochoric(cube)


def angles_deg(quaternions):
    return np.degrees(2.0 * np.arccos(np.clip(np.abs(quaternions[:, 0]), -1.0, 1.0)))


grid_deg = np.linspace(0.0, 180.0, 361)
analytic = (1.0 - np.cos(np.radians(grid_deg))) / np.pi * np.pi / 180.0

fig, ax = plt.subplots(figsize=(7.4, 4.2))
for sample, label in (
    (naive_q, "uniform Euler angles"),
    (corrected_q, r"Euler with $\cos\Phi$ uniform"),
    (cubochoric_q, "uniform cubochoric"),
):
    ax.hist(angles_deg(sample), bins=120, range=(0.0, 180.0), density=True,
            histtype="step", lw=1.6, label=label)
ax.plot(grid_deg, analytic, "k--", lw=1.4, label=r"analytic $(1-\cos\omega)/\pi$")
ax.set_xlabel("rotation angle (deg)")
ax.set_ylabel("probability density (per deg)")
ax.set_title("only two of these three samplers are uniform on SO(3)")
ax.legend()
plt.show()

analytic_mean = np.degrees(np.pi / 2.0 + 2.0 / np.pi)
print(f"analytic mean angle            : {analytic_mean:.4f} deg")
for sample, label in ((naive_q, "uniform Euler"), (corrected_q, "cos-Phi uniform"),
                      (cubochoric_q, "uniform cubochoric")):
    mean = float(angles_deg(sample).mean())
    print(f"  {label:<20}: {mean:8.4f} deg   (error {mean - analytic_mean:+.4f})")

The naive sampler is visibly biased towards small angles — it packs orientations near the
identity that a uniform distribution would not. The corrected Euler sampler works, but it
requires knowing to put $\cos\Phi$ uniform rather than $\Phi$, and the analogous
correction differs for every chart. The cubochoric sampler needs no correction at all,
because the chart itself is volume-preserving. That is the whole point of the next two
sections.

## 7. Homochoric coordinates: the equal-volume ball

Look for a radial function $f(\omega)$ such that the ordinary Euclidean volume element of
$\mathbf{h} = f(\omega)\,\hat{n}$ reproduces the Haar measure. Matching
$f^{2}\,\mathrm{d}f \propto (1-\cos\omega)\,\mathrm{d}\omega$ and normalising so that
$f(\omega) \approx \omega/2$ near the identity gives

$$\mathbf{h} = \hat{n}\left[\tfrac{3}{4}\left(\omega - \sin\omega\right)\right]^{1/3}.$$

All of $SO(3)$ then fills the ball of radius $R_1 = (3\pi/4)^{1/3} \approx 1.3307$, whose
volume is exactly $\pi^{2}$. Antipodal points of the bounding sphere are the *same*
rotation — a turn of $+\pi$ and of $-\pi$ about one axis agree — which is what makes the
ball a model of $SO(3)$ rather than of the quaternion sphere.

The inverse has no closed form: recovering $\omega$ means solving
$\omega - \sin\omega = \tfrac{4}{3}\|\mathbf{h}\|^{3}$. PyTex does it with a vectorized
bisection, sixty array operations regardless of batch size, rather than a per-element root
finder in a Python loop.

In [ ]:
angles_deg = np.linspace(0.0, 180.0, 500)
radii = np.cbrt(0.75 * (np.radians(angles_deg) - np.sin(np.radians(angles_deg))))

quaternions = np.column_stack([
    np.cos(np.radians(angles_deg) / 2.0),
    np.outer(np.sin(np.radians(angles_deg) / 2.0), np.array([0.0, 0.0, 1.0])),
])
computed = np.linalg.norm(homochoric_from_quaternions(quaternions), axis=1)

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.plot(angles_deg, radii, lw=3, alpha=0.35, label="analytic law")
ax.plot(angles_deg, computed, "k--", lw=1.2, label="homochoric_from_quaternions")
ax.plot(angles_deg, np.radians(angles_deg) / 2.0, ":", color="tab:red",
        label=r"small-angle limit $\omega/2$")
ax.set_xlabel("rotation angle (deg)")
ax.set_ylabel(r"$\|\mathbf{h}\|$")
ax.set_title("the homochoric radius, and its unit slope at the origin")
ax.legend()
plt.show()

print(f"max deviation from the analytic law: {float(np.abs(computed - radii).max()):.2e}")
print(f"radius at 180 deg: {computed[-1]:.9f}   R1 = {HOMOCHORIC_BALL_RADIUS:.9f}")

## 8. Cubochoric coordinates: equal volume *and* a cube

A ball is equal-volume but structurally awkward: a regular grid inside a ball is not a
regular grid of anything. Cubochoric coordinates map the ball onto a **cube** of the same
volume $\pi^{2}$, so of edge $a_p = \pi^{2/3} \approx 2.145$. A uniform Cartesian grid in
the cube is then a uniform grid of orientations, which is exactly what dictionary
indexing needs.

The map is that of Rosca, Morawiec and De Graef. PyTex derives it from two conditions
rather than quoting its constants, and both are worth stating because they explain
everything else:

1. **Nested surfaces.** The sub-cube of half-edge $z$ maps onto the sphere enclosing the
   same volume: $(2z)^3 = \tfrac{4}{3}\pi r^3$, so $r = z\,(6/\pi)^{1/3}$.
2. **Faces map to sectors, not caps.** The six face images must tile the sphere, and the
   map commutes with the octahedral symmetry, so the boundary between the $+z$ and $+x$
   images lies on the mirror plane $x = z$. Each face therefore maps to the curvilinear
   square $\{n_z \ge |n_x|, |n_y|\}$. Six spherical *caps* of the same solid angle would
   overlap and could not tile.

Together those force an area-preserving square-to-sector map, which factors into a planar
wedge and a Lambert azimuthal equal-area lift. The derivation is in
`docs/site/theory/orientation_representations.md`.

### 8.1 Condition 1, checked

In [ ]:
rng = np.random.default_rng(11)
fractions = (0.25, 0.5, 0.75, 1.0)

print("sub-cube half-edge -> sphere radius of its image")
for fraction in fractions:
    half = fraction * CUBOCHORIC_CUBE_HALF_EDGE
    faces = []
    for axis in range(3):
        for sign in (-1.0, 1.0):
            points = rng.uniform(-half, half, size=(300, 3))
            points[:, axis] = sign * half
            faces.append(points)
    image = convert_orientations(np.vstack(faces), source="cubochoric", target="homochoric")
    radii = np.linalg.norm(image, axis=1)
    expected = half * (6.0 / np.pi) ** (1.0 / 3.0)
    print(f"  {half:.5f}  ->  {radii.min():.9f} .. {radii.max():.9f}"
          f"   (predicted {expected:.9f})")

Every point of a cube *face* — corners, edge midpoints, face centres alike — lands on one
sphere, to nine decimals. That is the nested-surface law, and it is not something a
plausible-looking but wrong map would satisfy.

### 8.2 Volume preservation, checked

The defining property is that the Jacobian determinant of the map is $1$ **everywhere**,
not merely on average. Central differences at random interior points:

In [ ]:
def jacobian_determinant(point, step=1e-6):
    jacobian = np.empty((3, 3))
    for column in range(3):
        offset = np.zeros(3)
        offset[column] = step
        forward = convert_orientations((point + offset)[None, :],
                                       source="cubochoric", target="homochoric")[0]
        backward = convert_orientations((point - offset)[None, :],
                                        source="cubochoric", target="homochoric")[0]
        jacobian[:, column] = (forward - backward) / (2.0 * step)
    return float(np.linalg.det(jacobian))


probes = rng.uniform(-0.9, 0.9, size=(400, 3)) * CUBOCHORIC_CUBE_HALF_EDGE
determinants = np.array([jacobian_determinant(probe) for probe in probes])
print(f"|det J| over 400 interior points: "
      f"min {np.abs(determinants).min():.9f}, max {np.abs(determinants).max():.9f}")

### 8.3 What the map looks like

A regular grid on one face of the cube, and where it lands on the ball. The grid lines
bend, the cell *areas* do not.

In [ ]:
line = np.linspace(-1.0, 1.0, 17) * CUBOCHORIC_CUBE_HALF_EDGE
grid_x, grid_y = np.meshgrid(line, line, indexing="ij")
face = np.column_stack([
    grid_x.ravel(), grid_y.ravel(),
    np.full(grid_x.size, CUBOCHORIC_CUBE_HALF_EDGE),
])
image = convert_orientations(face, source="cubochoric", target="homochoric")

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.6))
axes[0].plot(grid_x, grid_y, color="tab:blue", lw=0.7)
axes[0].plot(grid_x.T, grid_y.T, color="tab:blue", lw=0.7)
axes[0].set_title("a regular grid on the cube's +z face")
axes[0].set_aspect("equal")

shape = (line.size, line.size)
axes[1].plot(image[:, 0].reshape(shape), image[:, 1].reshape(shape), color="tab:red", lw=0.7)
axes[1].plot(image[:, 0].reshape(shape).T, image[:, 1].reshape(shape).T,
             color="tab:red", lw=0.7)
circle = np.linspace(0.0, 2.0 * np.pi, 361)
axes[1].plot(HOMOCHORIC_BALL_RADIUS * np.cos(circle),
             HOMOCHORIC_BALL_RADIUS * np.sin(circle), "k-", lw=1.0)
axes[1].set_title("its image on the ball, viewed down z\n(black = the bounding sphere)")
axes[1].set_aspect("equal")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

print("the face's outer boundary lands on the bounding sphere:")
edge = np.abs(face[:, :2]).max(axis=1) > CUBOCHORIC_CUBE_HALF_EDGE - 1e-12
print(f"  |h| on the face edge: {np.linalg.norm(image[edge], axis=1).min():.9f} .. "
      f"{np.linalg.norm(image[edge], axis=1).max():.9f}")
print(f"  R1 = {HOMOCHORIC_BALL_RADIUS:.9f}")

The square becomes a curvilinear square filling one sixth of the disc's outline, meeting
its neighbours along the great circles at $45^\circ$ — condition 2 made visible. Note that
the boundary of the face maps to the bounding *sphere*, which is why the projected outline
touches the circle at the corners but is pulled inside it elsewhere: those points are out
of the plane.

## 9. All of it, in one call

`orientation_representations` converts once into every form and returns an object that can
print itself as a table, as convention-explicit prose, or as JSON. This is the answer to
"I have Euler angles, show me everything else".

In [ ]:
copper = Orientation.from_miller((1, 1, 2), (1, 1, -1), phase=NICKEL, specimen_frame=SPECIMEN)
report = orientation_representations(copper)
print(report.to_table())

In [ ]:
print(report.describe())

In [ ]:
payload = report.to_json_dict()
print("JSON keys:", sorted(payload))
print("\nschema:", payload["schema"])
print("ideal orientation payload:", payload["ideal_orientation"]["label"],
      "exact:", payload["ideal_orientation"]["is_exact"])

### Every field really is the same rotation

A report that merely *looked* consistent would be worthless. Rebuild the matrix from each
representation independently and compare.

In [ ]:
reference = copper.as_matrix()
rows = [
    (RepresentationKind.QUATERNION, report.quaternion),
    (RepresentationKind.AXIS_ANGLE, np.concatenate([report.axis, [np.radians(report.angle_deg)]])),
    (RepresentationKind.RODRIGUES, report.rodrigues),
    (RepresentationKind.RODRIGUES_FRANK, report.rodrigues_frank),
    (RepresentationKind.EULER_BUNGE, report.euler_bunge_deg),
    (RepresentationKind.EULER_MATTHIES, report.euler_matthies_deg),
    (RepresentationKind.HOMOCHORIC, report.homochoric),
    (RepresentationKind.CUBOCHORIC, report.cubochoric),
]
print(f"{'representation':<18}  max |rebuilt - g|")
for kind, values in rows:
    rebuilt = convert_orientations(np.asarray(values)[None, :], source=kind, target="matrix")[0]
    print(f"{kind.value:<18}  {float(np.abs(rebuilt - reference).max()):.3e}")

## 10. Naming an orientation: $(hkl)[uvw]$

Rolling texture is written in a language of names — cube, copper, brass, Goss — and each
name is a pair of index triples: $(hkl)$ is the crystal plane lying in the sheet plane,
$[uvw]$ the crystal direction along the rolling direction.

`ideal_orientation_indices` is the inverse of `Orientation.from_miller`. It maps ND and RD
back into the crystal, expresses them in the reciprocal and direct bases, and finds the
nearest integer triples — **and reports how far off they are**. That last part is not
decoration: only a measure-zero set of orientations has an exact ideal label, so a
component name quoted without its deviation is a claim the data does not support.

In [ ]:
components = {
    "cube":   ((0, 0, 1), (1, 0, 0)),
    "Goss":   ((0, 1, 1), (1, 0, 0)),
    "brass":  ((1, 1, 0), (1, -1, 2)),
    "copper": ((1, 1, 2), (1, 1, -1)),
    "S":      ((1, 2, 3), (6, 3, -4)),
}

print(f"{'component':<9} {'built from':<24} {'recovered':<24} {'phi1':>8} {'Phi':>7} {'phi2':>7}")
for name, (plane, direction) in components.items():
    orientation = Orientation.from_miller(
        plane, direction, phase=NICKEL, specimen_frame=SPECIMEN
    )
    indices = ideal_orientation_indices(orientation)
    euler = orientation.rotation.to_bunge_euler()
    built = f"{plane}{direction}".replace(" ", "")
    print(f"{name:<9} {built:<24} {indices.label:<24} "
          f"{euler[0]:8.2f} {euler[1]:7.2f} {euler[2]:7.2f}")

Every component round-trips: the indices come back exactly as they went in. Now perturb
one and watch the deviation appear rather than the label silently changing.

In [ ]:
exact = Orientation.from_miller((1, 1, 0), (1, -1, 2), phase=NICKEL, specimen_frame=SPECIMEN)

print(f"{'tilt (deg)':>10}  {'label':<22} {'plane dev':>10} {'dir dev':>9}  exact?")
for tilt_deg in (0.0, 0.5, 2.0, 5.0, 12.0):
    tilt = Rotation.from_axis_angle((0.0, 1.0, 0.0), np.radians(tilt_deg))
    perturbed = Orientation.from_matrix(
        tilt.as_matrix() @ exact.as_matrix(),
        specimen_frame=SPECIMEN, phase=NICKEL, crystal_frame=CRYSTAL,
    )
    indices = ideal_orientation_indices(perturbed, max_index=4)
    print(f"{tilt_deg:10.1f}  {indices.label:<22} "
          f"{indices.plane_deviation_deg:10.3f} {indices.direction_deviation_deg:9.3f}"
          f"  {indices.is_exact}")

The label stays "brass" through a 2-degree tilt and *says so* through its deviation. By 12
degrees the nearest label has changed to a different, larger-index triple — which is the
honest answer, and the reason the search bound `max_index` is a parameter rather than a
constant: raise it far enough and any orientation acquires an exact-looking name made of
meaningless indices.

## 11. The vectorized path

Every free function in `pytex.core.representations` takes a batch. The difference is not
cosmetic: the object-level Euler conversion in `RotationSet` builds one `Rotation` per row
in a Python loop, while `quaternions_from_euler_angles` does the same arithmetic in four
array operations.

In [ ]:
import time

rng = np.random.default_rng(4)
count = 20_000
angles = np.column_stack([
    rng.uniform(0.0, 360.0, count),
    np.degrees(np.arccos(rng.uniform(-1.0, 1.0, count))),
    rng.uniform(0.0, 360.0, count),
])

start = time.perf_counter()
vectorized = quaternions_from_euler_angles(angles, convention="bunge")
vectorized_seconds = time.perf_counter() - start

sample = angles[:2000]
start = time.perf_counter()
looped = np.stack([Rotation.from_bunge_euler(*triple).quaternion for triple in sample])
looped_seconds = time.perf_counter() - start

print(f"vectorized: {count} rotations in {vectorized_seconds * 1e3:.1f} ms")
print(f"per-object: {len(sample)} rotations in {looped_seconds * 1e3:.1f} ms")
print(f"extrapolated per-object cost for {count}: "
      f"{looped_seconds * count / len(sample) * 1e3:.0f} ms")
print(f"\nand the two agree exactly: "
      f"{float(np.abs(vectorized[:len(sample)] - looped).max()):.2e}")

`OrientationRepresentationSet` is the batch report: every representation of a whole
orientation cloud, each computed once for the batch rather than once per orientation.

In [ ]:
batch = OrientationRepresentationSet.from_values(angles[:5000], source="euler_bunge")
print(batch.describe())
print()
print("shapes:", {
    "quaternions": batch.quaternions.shape,
    "matrices": batch.matrices.shape,
    "rodrigues": batch.rodrigues.shape,
    "cubochoric": batch.cubochoric.shape,
})
print("\nrow 0 as a single report:")
print(batch.row(0).to_table())

## 12. Closure: every conversion, round-tripped

Ten representations, so ninety ordered pairs. Routing everything through quaternions means
there are ten conversions to maintain rather than ninety — and it means the whole set can
be checked at once.

In [ ]:
rng = np.random.default_rng(808)
cube = rng.uniform(-CUBOCHORIC_CUBE_HALF_EDGE, CUBOCHORIC_CUBE_HALF_EDGE, (512, 3))
reference = quaternions_from_cubochoric(cube)

kinds = list(RepresentationKind)
errors = np.zeros((len(kinds), len(kinds)))
for i, source in enumerate(kinds):
    values = convert_orientations(reference, source="quaternion", target=source)
    for j, target in enumerate(kinds):
        through = convert_orientations(values, source=source, target=target)
        back = convert_orientations(through, source=target, target="quaternion")
        alignment = np.abs(np.sum(back * reference, axis=1))
        errors[i, j] = float(np.max(np.abs(np.degrees(2.0 * np.arccos(np.clip(alignment, -1.0, 1.0))))))

fig, ax = plt.subplots(figsize=(7.2, 6.0))
image = ax.imshow(np.log10(np.maximum(errors, 1e-16)), cmap="viridis")
ax.set_xticks(range(len(kinds)), [k.value for k in kinds], rotation=90)
ax.set_yticks(range(len(kinds)), [k.value for k in kinds])
ax.set_xlabel("via"); ax.set_ylabel("from")
ax.set_title("round-trip closure error, log10(deg)\nquaternion -> from -> via -> quaternion")
fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()

print(f"worst closure error over all 100 paths: {errors.max():.3e} deg")

Every path closes to well below a millionth of a degree. Note that this is a *closure*
test, not an accuracy test — it would pass for a pair of mutually inverse but wrong maps.
The accuracy of the equal-volume maps is pinned separately, by the analytic checks in
sections 6-8 and by the property tests in
`tests/unit/test_orientation_representations.py`.

## 13. Failure modes, deliberately triggered

Three ways to get a wrong answer, and what PyTex does about each.

**(a) Rodrigues and homochoric vectors are both bare triples.** Nothing about the shape
distinguishes them, so a mix-up is silent — unless the domain is checked. The homochoric
ball is bounded, so PyTex can and does reject an out-of-ball input rather than clipping it.

In [ ]:
sharp = Rotation.from_axis_angle((1.0, 1.0, 1.0), np.radians(170.0))
rodrigues = sharp.to_rodrigues()
print("Rodrigues vector of a 170 deg rotation:", np.round(rodrigues, 4),
      f" norm {np.linalg.norm(rodrigues):.4f}")
print("the homochoric ball radius is only    ", f"{HOMOCHORIC_BALL_RADIUS:.4f}\n")

try:
    convert_orientations(rodrigues[None, :], source="homochoric", target="quaternion")
except ValueError as error:
    print("caught:", error)

**(b) The cubochoric cube is bounded too**, and by a *different* number. A coordinate
outside it is not a rotation at all.

In [ ]:
try:
    convert_orientations([[1.5, 0.0, 0.0]], source="cubochoric", target="quaternion")
except ValueError as error:
    print("caught:", error)

**(c) The quaternion sign is not information.** Comparing two quaternions component by
component will report a difference of 2 between a rotation and itself. Compare rotations,
not their coordinates.

In [ ]:
first = Rotation.from_bunge_euler(35.0, 45.0, 60.0)
flipped = Rotation(quaternion=-first.quaternion)

print("component-wise difference :", float(np.abs(flipped.quaternion - first.quaternion).max()))
print("angle between them        :", f"{np.degrees(first.distance_to(flipped)):.2e} deg")
print("matrix difference         :", float(np.abs(flipped.as_matrix() - first.as_matrix()).max()))

> **Good to know.**
>
> - Rodrigues wrote his vector down in 1840, three years before Hamilton's quaternions, and its
>   composition rule is Hamilton's product in disguise. Rodrigues-Frank space is still where
>   fundamental zones are polyhedra with flat faces, which is why texture codes keep it around.
> - Gimbal lock is a property of the chart, not of the rotation: nothing happens to the crystal at
>   $\Phi = 0$, but two of the three Euler angles stop being separately determined. It is famous
>   because Apollo 11's inertial platform had three gimbals and had to be flown around the
>   singularity.
> - Cubochoric coordinates (Roşca and Morawiec, 2014) are the modern answer to sampling: they map
>   the ball of rotations onto a cube while preserving volume, so a uniform grid in the cube is a
>   uniform grid in orientation space. Sampling Euler angles uniformly instead is the classic way
>   to over-sample the poles.

## 14. Choosing a representation

| you want to | use | why |
| --- | --- | --- |
| apply a rotation to vectors | matrix | it *is* the linear map |
| compose, interpolate, store | quaternion | no singularity, no convention branch, cheap product |
| read the physics off | axis-angle | Euler's theorem, directly |
| test membership of a fundamental zone | Rodrigues | the zone is a convex polyhedron |
| do the same near 180 degrees | Rodrigues-Frank | the tangent no longer diverges |
| exchange with EBSD or texture software | Bunge Euler | the community standard |
| exchange with Matthies/Roe sources | ZYZ Euler | what those sources quote |
| estimate a density in orientation space | homochoric | equal volume: no angle bias |
| build a uniform grid or sample | cubochoric | equal volume **and** a product structure |
| name a rolling-texture component | $(hkl)[uvw]$ | the language the literature uses |

Two rules survive from all of the above:

1. **Compose in quaternions; report in whatever the reader expects.** The equal-volume
   charts are charts, not algebras — never add two cubochoric coordinates.
2. **A triple of numbers is not an orientation until its convention, its frames, and its
   phase are attached.** That is why PyTex types carry all three, and why
   `describe()` prints them.

## Further reading

- `docs/site/theory/orientation_representations.md` — the full derivation, including the
  closed-form inverse of the cube-to-ball map.
- `docs/site/theory/euler_convention_handling.md` — the axis sequences and their aliases.
- `docs/site/theory/fundamental_region_reduction.md` — symmetry reduction, which none of
  this notebook does: every rotation here is a rotation, not an equivalence class.
- Tutorial 03, *Symmetry and fundamental regions* — where that reduction happens.
- Rosca, Morawiec and De Graef, *Modelling Simul. Mater. Sci. Eng.* **22** (2014) 075013 —
  the equal-volume map.
- Morawiec, *Orientations and Rotations*, Springer (2004) — the representation catalogue
  and the invariant measure.